# Stage 5 — Note causes, provisional validation and overtime attribution

Append after Stage 4; this stage requires only the Stage 1 dataframes. The breach
model, features and thresholds remain unchanged. No paid API or model download
is needed. Rules and conservative domain-word typo correction run locally.

## Taxonomy

| Category | Meaning | Reporting pile |
|---|---|---|
| client_requested | Explicit client-requested extra work | Client requested |
| absence_cover | Explicit cover for sickness, leave, absence or a named no-show | Operational associated |
| relief_problem | Replacement absent, late, or still unavailable | Operational associated |
| equipment_failure | Equipment/infrastructure trouble extends work | Operational associated |
| late_handover | Delay involving handover, keys, paperwork or transfer process | Operational associated |
| cover_unspecified | Cover mentioned without enough evidence of its cause | Unknown |
| unclear_or_mixed | Unmatched wording or multiple unresolved causes | Unknown |
| no_useful_information | Empty, placeholder or routine note | Unknown |

`relief_problem` deliberately includes late relief; `relief_no_show` would overstate
notes saying the replacement eventually arrived. Legitimate sickness or leave is
not employee wrongdoing. Operational-associated hours indicate coverage/operational
issues, not proven avoidable costs.

Client approval and root cause are separate. A client signing off cover for a
missing relief worker does not change the cause to client-requested additional work.
Approval reported by a note does not independently prove contractual billing rights.


In [1]:
import re, unicodedata, json
from pathlib import Path
from difflib import get_close_matches
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

required = ["notes", "shifts", "weekly_outcomes", "sites", "CURRENT_WEEK"]
assert all(name in globals() for name in required), "Run Stage 1 first."
assert notes.note.notna().all(), "Reload notes preserving empty strings and literal n/a."
assert notes.shift_id.is_unique
stage5_frozen_lr = lr_predictions.copy(deep=True) if "lr_predictions" in globals() else None
STAGE5_OUTPUT = Path("jem_stage5_outputs")
STAGE5_OUTPUT.mkdir(exist_ok=True)


## Reproducible classifier

Preserve the original note. Normalisation and typo corrections apply only to a
separate matching string. Correct only unambiguous close matches to a small domain
vocabulary; do not fuzzy-correct names or short words. Unknown wording is retained
as `unclear_or_mixed` rather than forced into a cause.

The review flag is a routing flag, not a probability of correctness. Multilingual
interpretations should be checked by a competent speaker. The rules below are
frozen for the reported provisional comparison; they have not been patched to
remove its four observed sample disagreements.


In [2]:
import re, unicodedata
from difflib import get_close_matches

CATEGORIES = ['client_requested','absence_cover','relief_problem','equipment_failure',
              'late_handover','cover_unspecified','unclear_or_mixed','no_useful_information']
CAUSE_PILES = {
 'client_requested':'client_requested',
 'absence_cover':'operational_associated',
 'relief_problem':'operational_associated',
 'equipment_failure':'operational_associated',
 'late_handover':'operational_associated',
 'cover_unspecified':'unknown', 'unclear_or_mixed':'unknown',
 'no_useful_information':'unknown',
}
ABBREVIATIONS={'agn':'again','hrs':'hours','mgr':'manager','mgmt':'management',
               'pls':'please','bc':'because'}
# Small domain vocabulary; names and short words are not fuzzy-corrected.
VOCABULARY=set('''client klient centre manager management requested request asked wanted
approved approval signed office extra relief replacement handover oorhandiging
waiting waited delayed paperwork nobody never arrived supposed machine scrubber
buffer generator motor failed fault manually broken stocktake covering covered
absent clinic responsibility leave sick akezanga akafikanga aflos opgedaag
stukkend masjien sleutels gevra gedek gemeld normal incidents nothing report'''.split())
ROUTINE={'','n a','na','ntr','ok','fine','sharp','quiet shift','all quiet','all good',
         'all fine','nothing to report','no incidents','no issues on site','as per normal',
         'akukho lutho','niks om te rapporteer nie'}

def normalise_note(text):
    text=unicodedata.normalize('NFKD',str(text)).encode('ascii','ignore').decode().lower()
    text=text.replace("'",'').replace('’','')
    text=re.sub(r'[^a-z0-9\s]',' ',text)
    words=[ABBREVIATIONS.get(word,word) for word in text.split()]
    return ' '.join(words)

def correct_domain_typos(text):
    corrected=[]; changes=[]
    for word in text.split():
        if word not in VOCABULARY and len(word)>=5 and word.isalpha():
            matches=get_close_matches(word,sorted(VOCABULARY),n=2,cutoff=0.88)
            # Ambiguous corrections are left alone.
            if len(matches)==1:
                corrected.append(matches[0]);changes.append(word+'→'+matches[0]);continue
        corrected.append(word)
    return ' '.join(corrected), '; '.join(changes)

def classify_note(text):
    original_normalised=normalise_note(text)
    clean,corrections=correct_domain_typos(original_normalised)
    def has(pattern):return re.search(pattern,clean) is not None
    language_flag=has(r'\b(aflos|oorhandiging|masjien|stukkend|klient|akezanga|akafikanga|ngimele|ngicela|gedek|siek|akukho|niks)\b')
    client_actor=has(r'\b(client|klient)\b|centre (manager|management)|site manager')
    request=has(r'\b(asked|requested|wanted|gevra)\b|says stay')
    client_request=client_actor and (request or has(r'extra hours approved by client'))
    uncertain_approval=has(r'dont know|do not know|not sure|awaiting approval|not approved|no approval')
    confirmed_approval=has(r'\bapproved\b|signed (off|for)|\bokd\b')
    approval='unconfirmed' if uncertain_approval else ('confirmed_in_note' if confirmed_approval else 'not_stated')

    matches=[]
    if has(r'\brelief\b|no replacement|next shift|\baflos\b'):
        # Require evidence of disrupted cover, not a bare mention of the word relief.
        if has(r'no |never|nobody|did not|didnt|only arrived|still on site|suppose|supposed|akafikanga|nie opgedaag|stayed|coming'):
            matches.append('relief_problem')
    if has(r'\babsent\b|off sick|at the clinic|family responsibility leave|didnt come|did not come|no show no call|\bakezanga\b|siek gemeld'):
        matches.append('absence_cover')
    equipment=has(r'\b(machine|scrubber|buffer|generator|masjien)\b|gate motor|\blift\b')
    failure=has(r'\b(broke|broken|down|fault|failed|kaput|stukkend)\b|out of order|by hand|manually')
    if equipment and failure and not has(r'not broken|no fault|no equipment problems'):
        matches.append('equipment_failure')
    if has(r'\bhandover\b|\boorhandiging\b|keys missing|ob book not signed') and has(r'late|delayed|wait|\blaat\b'):
        matches.append('late_handover')
    cover=has(r'stood in for|covered for|\bcovering\b|shift as well|2 posts 1 guard')
    if original_normalised in ROUTINE or clean in ROUTINE:
        category='no_useful_information';evidence='empty/placeholder/routine statement'
    elif len(matches)>1:
        category='unclear_or_mixed';evidence='multiple operational signals: '+', '.join(matches)
    elif len(matches)==1:
        if client_request and not has(r'real reason|\bbecause\b'):
            category='unclear_or_mixed';evidence='explicit client request and operational cause'
        else:
            category=matches[0];evidence='operational cause; approval does not replace cause'
    elif client_request:
        category='client_requested';evidence='client actor and explicit request/approved extra work'
    elif cover:
        category='cover_unspecified';evidence='cover mentioned without an explicit cause'
    else:
        category='unclear_or_mixed';evidence='no sufficiently specific rule matched'
    return {'category':category,'cause_pile':CAUSE_PILES[category],
            'explicit_client_request':bool(client_request),'approval_status':approval,
            'matched_evidence':evidence,'normalised_note':clean,'typo_corrections':corrections,
            'language_review_needed':bool(language_flag),
            'needs_review':bool(category in ['unclear_or_mixed','cover_unspecified'] or corrections or language_flag)}


In [3]:
classified = pd.DataFrame([classify_note(note) for note in notes.note])
note_classification_audit = pd.concat([notes.reset_index(drop=True), classified], axis=1)
note_classifications = note_classification_audit[["shift_id", "category", "note"]].copy()
assert list(note_classifications.columns) == ["shift_id", "category", "note"]
assert len(note_classifications) == len(notes) and note_classifications.shift_id.is_unique
assert set(note_classifications.shift_id) == set(notes.shift_id)
assert note_classifications.note.tolist() == notes.note.tolist()
assert note_classifications.category.isin(CATEGORIES).all()

# Material precedence and ambiguity checks, separate from sampled agreement.
case = classify_note("client signed for the extra hours but real reason is relief no show again")
assert case["category"] == "relief_problem" and case["approval_status"] == "confirmed_in_note"
case = classify_note("client says stay till six, dont know if office approved")
assert case["category"] == "client_requested" and case["approval_status"] == "unconfirmed"
assert classify_note("stood in for Sibiya")["cause_pile"] == "unknown"
assert classify_note("n/a")["category"] == "no_useful_information"
assert classify_note("")["category"] == "no_useful_information"
assert classify_note("scrubber broke down, had to do the floor manually")["category"] == "equipment_failure"

display(note_classification_audit.category.value_counts().rename("notes").to_frame())
display(pd.crosstab(note_classification_audit.cause_pile, note_classification_audit.approval_status))
print("Notes requiring some review:", int(note_classification_audit.needs_review.sum()))
print("Language-review flag:", int(note_classification_audit.language_review_needed.sum()))
display(note_classification_audit.loc[note_classification_audit.category.eq("unclear_or_mixed"),
    ["shift_id", "note", "matched_evidence"]].head(15))


                       notes
category                    
no_useful_information    435
client_requested         391
absence_cover            380
relief_problem           297
late_handover            202
equipment_failure        188
cover_unspecified        133
unclear_or_mixed          91
approval_status         confirmed_in_note  not_stated  unconfirmed
cause_pile                                                        
client_requested                      227         135           29
operational_associated                 19        1048            0
unknown                                37         620            2
Notes requiring some review: 633
Language-review flag: 284
    shift_id                                               note  \
10   S108056    Nexxt shift guuard did not pitch. Had to cover.   
20   S106113  etra hours approved by client for the event setup   
59   S106878           September didt come in, covered the post   
120  S104802  extra patrol per client email, app

## Provisional reference sample — provenance matters

The supplied export was sampled by row: 50 development notes (seed 104), then
100 random validation notes from the remaining rows (seed 205). A separate set
of 20 challenge notes (seed 306) was drawn from the remaining rows using wording
about approval conflicts, multilingual notes and unspecified cover. Challenge
results are not a population estimate.

The embedded reference categories were assigned by AI reading those texts before
executing the classifier. They are NOT independent human ground truth. The same
assistant had already inspected recurring note templates and designed the rules;
this is therefore a provisional consistency check, not a pristine blind test.
Repeated templates also limit what high agreement can establish.

The exact original texts are embedded to prevent stale labels being applied to
changed notes on a later export. The validation code evaluates only matching IDs
and identical text. The human-review CSV hides both predictions and AI categories.
Independent human review, especially of multilingual wording, remains outstanding.


In [4]:
AI_REFERENCE_ROWS = json.loads("[{\"shift_id\": \"S101540\", \"logged_by\": \"SUP-23\", \"note\": \"relief never arriived, stayed on till 6am..\", \"sample_split\": \"development\", \"sample_id\": \"N001\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107980\", \"logged_by\": \"SUP-11\", \"note\": \"control room sayys relief coming, noboddy came\", \"sample_split\": \"development\", \"sample_id\": \"N002\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101987\", \"logged_by\": \"SUP-14\", \"note\": \"quiet shift\", \"sample_split\": \"development\", \"sample_id\": \"N003\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108183\", \"logged_by\": \"SUP-08\", \"note\": \"double duty today, Radebe on family responsibility leave\", \"sample_split\": \"development\", \"sample_id\": \"N004\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101047\", \"logged_by\": \"SUP-10\", \"note\": \"Centre manager requested additional cover for stocktake\", \"sample_split\": \"development\", \"sample_id\": \"N005\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102079\", \"logged_by\": \"SUP-03\", \"note\": \"all good\", \"sample_split\": \"development\", \"sample_id\": \"N006\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100845\", \"logged_by\": \"SUP-07\", \"note\": \"n/a\", \"sample_split\": \"development\", \"sample_id\": \"N007\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101392\", \"logged_by\": \"SUP-24\", \"note\": \"handover late again, keys missing\", \"sample_split\": \"development\", \"sample_id\": \"N008\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108442\", \"logged_by\": \"SUP-08\", \"note\": \"oorhandiging was laat, gewag vir sleutels\", \"sample_split\": \"development\", \"sample_id\": \"N009\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106578\", \"logged_by\": \"SUP-24\", \"note\": \"centre manager requested additional cover for stocktake !\", \"sample_split\": \"development\", \"sample_id\": \"N010\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101577\", \"logged_by\": \"SUP-17\", \"note\": \"stocktake ran over, client asked us to remain, they know they pay for it\", \"sample_split\": \"development\", \"sample_id\": \"N011\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102428\", \"logged_by\": \"SUP-05\", \"note\": \"stocktake ran over, client asked us to remain, they know they pay for it\", \"sample_split\": \"development\", \"sample_id\": \"N012\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103762\", \"logged_by\": \"SUP-10\", \"note\": \"Ngcobo didnt come in, covered the post\", \"sample_split\": \"development\", \"sample_id\": \"N013\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108026\", \"logged_by\": \"SUP-11\", \"note\": \"srubber broke down, had to do the floor manually\", \"sample_split\": \"development\", \"sample_id\": \"N014\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100391\", \"logged_by\": \"SUP-14\", \"note\": \"aflos het nie opgedaag nie, moes aabnly\", \"sample_split\": \"development\", \"sample_id\": \"N015\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106151\", \"logged_by\": \"SUP-14\", \"note\": \"shift handover delayed by 50 min\", \"sample_split\": \"development\", \"sample_id\": \"N016\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104171\", \"logged_by\": \"SUP-14\", \"note\": \"requested by centre management for load in, signed off\", \"sample_split\": \"development\", \"sample_id\": \"N017\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100880\", \"logged_by\": \"SUP-05\", \"note\": \"Ndlovu absent, took her rounds as well\", \"sample_split\": \"development\", \"sample_id\": \"N018\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108615\", \"logged_by\": \"SUP-16\", \"note\": \"wyk off sick. again. covered..\", \"sample_split\": \"development\", \"sample_id\": \"N019\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105761\", \"logged_by\": \"SUP-16\", \"note\": \"scrubber broke down, had to do the floor manually !\", \"sample_split\": \"development\", \"sample_id\": \"N020\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106247\", \"logged_by\": \"SUP-01\", \"note\": \"next shift guard did not pitch. had to cover\", \"sample_split\": \"development\", \"sample_id\": \"N021\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107977\", \"logged_by\": \"SUP-23\", \"note\": \"late hanover, wating on paperwork\", \"sample_split\": \"development\", \"sample_id\": \"N022\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101765\", \"logged_by\": \"SUP-12\", \"note\": \"all good\", \"sample_split\": \"development\", \"sample_id\": \"N023\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102011\", \"logged_by\": \"SUP-17\", \"note\": \"all good\", \"sample_split\": \"development\", \"sample_id\": \"N024\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106790\", \"logged_by\": \"SUP-22\", \"note\": \"as per normal\", \"sample_split\": \"development\", \"sample_id\": \"N025\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106096\", \"logged_by\": \"SUP-18\", \"note\": \"ufourie akezanga namhlanje, ngimele yena\", \"sample_split\": \"development\", \"sample_id\": \"N026\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106462\", \"logged_by\": \"SUP-21\", \"note\": \"stocktake ran over, client asked us to remain, they know they pay for it\", \"sample_split\": \"development\", \"sample_id\": \"N027\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100585\", \"logged_by\": \"SUP-11\", \"note\": \"extra hrs approved by client for the event setup\", \"sample_split\": \"development\", \"sample_id\": \"N028\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104748\", \"logged_by\": \"SUP-15\", \"note\": \"next shift akafikanga, ngihlale kuze kube 6..\", \"sample_split\": \"development\", \"sample_id\": \"N029\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108314\", \"logged_by\": \"SUP-06\", \"note\": \"no issues on site\", \"sample_split\": \"development\", \"sample_id\": \"N030\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103033\", \"logged_by\": \"SUP-16\", \"note\": \"aflos het nie opgedaag nie, moes aanbly\", \"sample_split\": \"development\", \"sample_id\": \"N031\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100717\", \"logged_by\": \"SUP-07\", \"note\": \"next shift akafikanga, ngihlale kuze kube 6am\", \"sample_split\": \"development\", \"sample_id\": \"N032\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105735\", \"logged_by\": \"SUP-12\", \"note\": \"worked through, sibiya at the clinic\", \"sample_split\": \"development\", \"sample_id\": \"N033\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107529\", \"logged_by\": \"SUP-07\", \"note\": \"worked through, Zulu at the clinic\", \"sample_split\": \"development\", \"sample_id\": \"N034\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103710\", \"logged_by\": \"SUP-09\", \"note\": \"shift handover delayed by 20 min\", \"sample_split\": \"development\", \"sample_id\": \"N035\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106935\", \"logged_by\": \"SUP-15\", \"note\": \"stocktake ran over, client asked us to remain, they knnow they pay for it\", \"sample_split\": \"development\", \"sample_id\": \"N036\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101898\", \"logged_by\": \"SUP-22\", \"note\": \"Sibiya didnt come in, covered the post\", \"sample_split\": \"development\", \"sample_id\": \"N037\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S109246\", \"logged_by\": \"SUP-07\", \"note\": \"fine\", \"sample_split\": \"development\", \"sample_id\": \"N038\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102535\", \"logged_by\": \"SUP-06\", \"note\": \"quiet shift\", \"sample_split\": \"development\", \"sample_id\": \"N039\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101704\", \"logged_by\": \"SUP-11\", \"note\": \"buffer machine kaput, did the floor by hand !\", \"sample_split\": \"development\", \"sample_id\": \"N040\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103775\", \"logged_by\": \"SUP-03\", \"note\": \"scrubber broke down, had to do the floor manually..\", \"sample_split\": \"development\", \"sample_id\": \"N041\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104883\", \"logged_by\": \"SUP-14\", \"note\": \"ok\", \"sample_split\": \"development\", \"sample_id\": \"N042\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101099\", \"logged_by\": \"SUP-03\", \"note\": \"oorhandiging was laat, gewag vir sleutels\", \"sample_split\": \"development\", \"sample_id\": \"N043\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105334\", \"logged_by\": \"SUP-11\", \"note\": \"client asked us to stay for the delivery, ok'd by centre mgmt\", \"sample_split\": \"development\", \"sample_id\": \"N044\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107322\", \"logged_by\": \"SUP-10\", \"note\": \"client says stay till 06h00, dont know if office apprved\", \"sample_split\": \"development\", \"sample_id\": \"N045\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106048\", \"logged_by\": \"SUP-07\", \"note\": \"stocktake ran over, client asked us to remain, they know thy pay for it\", \"sample_split\": \"development\", \"sample_id\": \"N046\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105834\", \"logged_by\": \"SUP-09\", \"note\": \"generator fault, stayed to monitor\", \"sample_split\": \"development\", \"sample_id\": \"N047\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104568\", \"logged_by\": \"SUP-03\", \"note\": \"Cele absent, took her rounds as well\", \"sample_split\": \"development\", \"sample_id\": \"N048\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107344\", \"logged_by\": \"SUP-08\", \"note\": \"handover late agn, keys missing\", \"sample_split\": \"development\", \"sample_id\": \"N049\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106955\", \"logged_by\": \"SUP-21\", \"note\": \"maluleke off sick. again. covered.\", \"sample_split\": \"development\", \"sample_id\": \"N050\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107549\", \"logged_by\": \"SUP-06\", \"note\": \"\", \"sample_split\": \"validation\", \"sample_id\": \"N051\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104893\", \"logged_by\": \"SUP-01\", \"note\": \"waited 20 min for handover, OB book not signed\", \"sample_split\": \"validation\", \"sample_id\": \"N052\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107746\", \"logged_by\": \"SUP-21\", \"note\": \"still on site, relief was suppose to come 06h00\", \"sample_split\": \"validation\", \"sample_id\": \"N053\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104116\", \"logged_by\": \"SUP-06\", \"note\": \"uMaluleke akezanga namhlanje, ngimele yena\", \"sample_split\": \"validation\", \"sample_id\": \"N054\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102601\", \"logged_by\": \"SUP-18\", \"note\": \"Client asked for extra ptrol after the break-in Tuesday\", \"sample_split\": \"validation\", \"sample_id\": \"N055\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104188\", \"logged_by\": \"SUP-09\", \"note\": \"TOOK SIBIYA SHIFT AS WELL, 2 POSTS 1 GUARD\", \"sample_split\": \"validation\", \"sample_id\": \"N056\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103796\", \"logged_by\": \"SUP-07\", \"note\": \"no relief. stayed. someone must please sort the roster\", \"sample_split\": \"validation\", \"sample_id\": \"N057\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102134\", \"logged_by\": \"SUP-08\", \"note\": \"handover late agn, keys missing !\", \"sample_split\": \"validation\", \"sample_id\": \"N058\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101758\", \"logged_by\": \"SUP-14\", \"note\": \"Client requested deep clean before the audit - approved\", \"sample_split\": \"validation\", \"sample_id\": \"N059\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101589\", \"logged_by\": \"SUP-03\", \"note\": \"conntrol room says relief coming, nobody came\", \"sample_split\": \"validation\", \"sample_id\": \"N060\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106806\", \"logged_by\": \"SUP-20\", \"note\": \"Naidoo didnt come in, covered the post..\", \"sample_split\": \"validation\", \"sample_id\": \"N061\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106113\", \"logged_by\": \"SUP-20\", \"note\": \"etra hours approved by client for the event setup\", \"sample_split\": \"validation\", \"sample_id\": \"N062\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107077\", \"logged_by\": \"SUP-04\", \"note\": \"stood in for Sibiya\", \"sample_split\": \"validation\", \"sample_id\": \"N063\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100406\", \"logged_by\": \"SUP-01\", \"note\": \"as per normal\", \"sample_split\": \"validation\", \"sample_id\": \"N064\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108377\", \"logged_by\": \"SUP-20\", \"note\": \"extra hours approved by client for the event setup\", \"sample_split\": \"validation\", \"sample_id\": \"N065\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103411\", \"logged_by\": \"SUP-08\", \"note\": \"ceentre manager rqeuested additional cover for stocktake\", \"sample_split\": \"validation\", \"sample_id\": \"N066\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101222\", \"logged_by\": \"SUP-20\", \"note\": \"all good\", \"sample_split\": \"validation\", \"sample_id\": \"N067\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101609\", \"logged_by\": \"SUP-02\", \"note\": \"covering Cele post, no show no call\", \"sample_split\": \"validation\", \"sample_id\": \"N068\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100721\", \"logged_by\": \"SUP-18\", \"note\": \"oorhandiging was laat, gewag vir slutels\", \"sample_split\": \"validation\", \"sample_id\": \"N069\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103752\", \"logged_by\": \"SUP-21\", \"note\": \"stood in for Ndlovu\", \"sample_split\": \"validation\", \"sample_id\": \"N070\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105204\", \"logged_by\": \"SUP-10\", \"note\": \"buffer machine kaput, did the floor by hand\", \"sample_split\": \"validation\", \"sample_id\": \"N071\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105653\", \"logged_by\": \"SUP-24\", \"note\": \"stood in for Jacobs\", \"sample_split\": \"validation\", \"sample_id\": \"N072\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101110\", \"logged_by\": \"SUP-13\", \"note\": \"extra hrs approved by client for the event settup\", \"sample_split\": \"validation\", \"sample_id\": \"N073\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100329\", \"logged_by\": \"SUP-18\", \"note\": \"client signed for the extra hours but real reason is relief no show again\", \"sample_split\": \"validation\", \"sample_id\": \"N074\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100693\", \"logged_by\": \"SUP-04\", \"note\": \"handover late agn, keys missing !\", \"sample_split\": \"validation\", \"sample_id\": \"N075\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103055\", \"logged_by\": \"SUP-17\", \"note\": \"stood in for Fourie\", \"sample_split\": \"validation\", \"sample_id\": \"N076\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104444\", \"logged_by\": \"SUP-21\", \"note\": \"relief only arrived six, stayed until then\", \"sample_split\": \"validation\", \"sample_id\": \"N077\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101230\", \"logged_by\": \"SUP-20\", \"note\": \"ntr\", \"sample_split\": \"validation\", \"sample_id\": \"N078\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S109349\", \"logged_by\": \"SUP-05\", \"note\": \"n/a\", \"sample_split\": \"validation\", \"sample_id\": \"N079\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103413\", \"logged_by\": \"SUP-24\", \"note\": \"-\", \"sample_split\": \"validation\", \"sample_id\": \"N080\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103011\", \"logged_by\": \"SUP-02\", \"note\": \"still on site, relief was suppose to coe six\", \"sample_split\": \"validation\", \"sample_id\": \"N081\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100926\", \"logged_by\": \"SUP-11\", \"note\": \"covering Hendricks pos, no show no call\", \"sample_split\": \"validation\", \"sample_id\": \"N082\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101748\", \"logged_by\": \"SUP-16\", \"note\": \"Centre manager requested additional cover for stocktake\", \"sample_split\": \"validation\", \"sample_id\": \"N083\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102020\", \"logged_by\": \"SUP-24\", \"note\": \"took Wyk shift as well, 2 posts 1 guard\", \"sample_split\": \"validation\", \"sample_id\": \"N084\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106107\", \"logged_by\": \"SUP-08\", \"note\": \"machine down again, took twice as long\", \"sample_split\": \"validation\", \"sample_id\": \"N085\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104295\", \"logged_by\": \"SUP-01\", \"note\": \"machine down again, took twice as long\", \"sample_split\": \"validation\", \"sample_id\": \"N086\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105011\", \"logged_by\": \"SUP-11\", \"note\": \"Client requested deep clean before the audit - approved\", \"sample_split\": \"validation\", \"sample_id\": \"N087\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100543\", \"logged_by\": \"SUP-03\", \"note\": \"gate motor failed, manned it by hand till 0600..\", \"sample_split\": \"validation\", \"sample_id\": \"N088\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106646\", \"logged_by\": \"SUP-16\", \"note\": \"client requested deep clean befroe the audit - approved\", \"sample_split\": \"validation\", \"sample_id\": \"N089\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108830\", \"logged_by\": \"SUP-21\", \"note\": \"requested by cenre management for load in, siged off\", \"sample_split\": \"validation\", \"sample_id\": \"N090\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108017\", \"logged_by\": \"SUP-19\", \"note\": \"relief only arrived six, stayed until thhen\", \"sample_split\": \"validation\", \"sample_id\": \"N091\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100277\", \"logged_by\": \"SUP-18\", \"note\": \"shift handover delayed by 20 min\", \"sample_split\": \"validation\", \"sample_id\": \"N092\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108863\", \"logged_by\": \"SUP-09\", \"note\": \"double duty today, sithole on family responsibility leave\", \"sample_split\": \"validation\", \"sample_id\": \"N093\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103338\", \"logged_by\": \"SUP-20\", \"note\": \"late hanodver, waiting on paperwork\", \"sample_split\": \"validation\", \"sample_id\": \"N094\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102704\", \"logged_by\": \"SUP-20\", \"note\": \"ntr\", \"sample_split\": \"validation\", \"sample_id\": \"N095\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106859\", \"logged_by\": \"SUP-11\", \"note\": \"fine\", \"sample_split\": \"validation\", \"sample_id\": \"N096\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106542\", \"logged_by\": \"SUP-06\", \"note\": \"covering Ndlovu post, no show no call\", \"sample_split\": \"validation\", \"sample_id\": \"N097\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S109299\", \"logged_by\": \"SUP-01\", \"note\": \"Hendricks didnt come in, coevred the post\", \"sample_split\": \"validation\", \"sample_id\": \"N098\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107298\", \"logged_by\": \"SUP-08\", \"note\": \"double duty today, naidoo on fammily responsibility leave..\", \"sample_split\": \"validation\", \"sample_id\": \"N099\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104081\", \"logged_by\": \"SUP-07\", \"note\": \"no replacement sent, ngicela sort this out\", \"sample_split\": \"validation\", \"sample_id\": \"N100\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103062\", \"logged_by\": \"SUP-24\", \"note\": \"no replacement sent, ngicela sort this out\", \"sample_split\": \"validation\", \"sample_id\": \"N101\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105004\", \"logged_by\": \"SUP-16\", \"note\": \"Maluleke didnt come in, covered the pot..\", \"sample_split\": \"validation\", \"sample_id\": \"N102\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101813\", \"logged_by\": \"SUP-11\", \"note\": \"ntr\", \"sample_split\": \"validation\", \"sample_id\": \"N103\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104010\", \"logged_by\": \"SUP-10\", \"note\": \"took Ndlovu shift as well, 2 psots 1 guard\", \"sample_split\": \"validation\", \"sample_id\": \"N104\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105585\", \"logged_by\": \"SUP-12\", \"note\": \"scrubber broke dwn, had to do the floor mnually\", \"sample_split\": \"validation\", \"sample_id\": \"N105\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107972\", \"logged_by\": \"SUP-10\", \"note\": \"client asked us to stay for the delivery, okd' by centre mgmt\", \"sample_split\": \"validation\", \"sample_id\": \"N106\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106818\", \"logged_by\": \"SUP-06\", \"note\": \".\", \"sample_split\": \"validation\", \"sample_id\": \"N107\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105207\", \"logged_by\": \"SUP-02\", \"note\": \"sharp\", \"sample_split\": \"validation\", \"sample_id\": \"N108\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107610\", \"logged_by\": \"SUP-08\", \"note\": \"ok\", \"sample_split\": \"validation\", \"sample_id\": \"N109\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107360\", \"logged_by\": \"SUP-09\", \"note\": \"NDLOVU OFF SICK. AGN. COVERED.\", \"sample_split\": \"validation\", \"sample_id\": \"N110\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102287\", \"logged_by\": \"SUP-12\", \"note\": \"extra hrs approved by client for the event setup\", \"sample_split\": \"validation\", \"sample_id\": \"N111\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106433\", \"logged_by\": \"SUP-15\", \"note\": \"extra hrs approved by client for the event setup\", \"sample_split\": \"validation\", \"sample_id\": \"N112\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107524\", \"logged_by\": \"SUP-19\", \"note\": \"double duty today, Sibiya on family responsibility leave\", \"sample_split\": \"validation\", \"sample_id\": \"N113\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103850\", \"logged_by\": \"SUP-16\", \"note\": \"no issues on site\", \"sample_split\": \"validation\", \"sample_id\": \"N114\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108405\", \"logged_by\": \"SUP-10\", \"note\": \"gate motor failed, manned it by hand till 06:00\", \"sample_split\": \"validation\", \"sample_id\": \"N115\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106556\", \"logged_by\": \"SUP-17\", \"note\": \"gate motor failed, manned it by hand till six\", \"sample_split\": \"validation\", \"sample_id\": \"N116\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106334\", \"logged_by\": \"SUP-22\", \"note\": \"Client asked for extra patrol aftter the break-in Tuesday\", \"sample_split\": \"validation\", \"sample_id\": \"N117\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102752\", \"logged_by\": \"SUP-03\", \"note\": \"requested by centre management for load in, signed off\", \"sample_split\": \"validation\", \"sample_id\": \"N118\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102395\", \"logged_by\": \"SUP-01\", \"note\": \"aflos het nie opgedaag nie, moes aanbly\", \"sample_split\": \"validation\", \"sample_id\": \"N119\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106814\", \"logged_by\": \"SUP-01\", \"note\": \"Mabaso didnt come in, covered the post\", \"sample_split\": \"validation\", \"sample_id\": \"N120\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100250\", \"logged_by\": \"SUP-03\", \"note\": \"relief onnly arrived sixx, styed until then\", \"sample_split\": \"validation\", \"sample_id\": \"N121\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103447\", \"logged_by\": \"SUP-10\", \"note\": \"relief never arrived, stayed on till 06:00\", \"sample_split\": \"validation\", \"sample_id\": \"N122\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107118\", \"logged_by\": \"SUP-20\", \"note\": \"klient het ekstra ure gevra vir stocktake\", \"sample_split\": \"validation\", \"sample_id\": \"N123\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107901\", \"logged_by\": \"SUP-09\", \"note\": \"late handover, waiting on paperwork\", \"sample_split\": \"validation\", \"sample_id\": \"N124\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108065\", \"logged_by\": \"SUP-10\", \"note\": \"Jacobs didnt come in, covered the post\", \"sample_split\": \"validation\", \"sample_id\": \"N125\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105352\", \"logged_by\": \"SUP-24\", \"note\": \"client says stay till six, dont know if office approved\", \"sample_split\": \"validation\", \"sample_id\": \"N126\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102288\", \"logged_by\": \"SUP-02\", \"note\": \"extra hrs approved by client for the event setup\", \"sample_split\": \"validation\", \"sample_id\": \"N127\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107006\", \"logged_by\": \"SUP-03\", \"note\": \"clieent asked us to stay for the delivery, ok'd by centre mgmt\", \"sample_split\": \"validation\", \"sample_id\": \"N128\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108483\", \"logged_by\": \"SUP-14\", \"note\": \"Covering for Molefe - booked off sick\", \"sample_split\": \"validation\", \"sample_id\": \"N129\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100114\", \"logged_by\": \"SUP-15\", \"note\": \"waited 25 min for handover, ob book not signed\", \"sample_split\": \"validation\", \"sample_id\": \"N130\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107088\", \"logged_by\": \"SUP-06\", \"note\": \"additional cover requested by site mgr, signed off\", \"sample_split\": \"validation\", \"sample_id\": \"N131\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107175\", \"logged_by\": \"SUP-15\", \"note\": \"no incidents\", \"sample_split\": \"validation\", \"sample_id\": \"N132\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100171\", \"logged_by\": \"SUP-11\", \"note\": \"Client asked for extra patrol after the break-in Tuesday..\", \"sample_split\": \"validation\", \"sample_id\": \"N133\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105472\", \"logged_by\": \"SUP-22\", \"note\": \"-\", \"sample_split\": \"validation\", \"sample_id\": \"N134\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100338\", \"logged_by\": \"SUP-24\", \"note\": \"client wanted extra man on the gate for the event, approved\", \"sample_split\": \"validation\", \"sample_id\": \"N135\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100849\", \"logged_by\": \"SUP-06\", \"note\": \"no issues on site\", \"sample_split\": \"validation\", \"sample_id\": \"N136\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100337\", \"logged_by\": \"SUP-10\", \"note\": \"Centre mgr requested additional cover for stocktake\", \"sample_split\": \"validation\", \"sample_id\": \"N137\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100817\", \"logged_by\": \"SUP-15\", \"note\": \"all good\", \"sample_split\": \"validation\", \"sample_id\": \"N138\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103561\", \"logged_by\": \"SUP-13\", \"note\": \"toook motaung shift as well, 2 posts 1 guard\", \"sample_split\": \"validation\", \"sample_id\": \"N139\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S105318\", \"logged_by\": \"SUP-16\", \"note\": \"client signed for the extra hours but real reason is relief no show agn !\", \"sample_split\": \"validation\", \"sample_id\": \"N140\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102050\", \"logged_by\": \"SUP-09\", \"note\": \"gedk vir Sithole, siek gemeld\", \"sample_split\": \"validation\", \"sample_id\": \"N141\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108835\", \"logged_by\": \"SUP-08\", \"note\": \"client wanted extra man on the gate for the event, approved\", \"sample_split\": \"validation\", \"sample_id\": \"N142\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106973\", \"logged_by\": \"SUP-18\", \"note\": \"client asked us to stay for the delivery, ok'd by centre mgmt..\", \"sample_split\": \"validation\", \"sample_id\": \"N143\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107724\", \"logged_by\": \"SUP-09\", \"note\": \"stocktake ran over, client asked us to remain, they know they pay for it\", \"sample_split\": \"validation\", \"sample_id\": \"N144\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103565\", \"logged_by\": \"SUP-17\", \"note\": \"ntr\", \"sample_split\": \"validation\", \"sample_id\": \"N145\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106218\", \"logged_by\": \"SUP-19\", \"note\": \"covering Naido post, no show no call\", \"sample_split\": \"validation\", \"sample_id\": \"N146\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107100\", \"logged_by\": \"SUP-01\", \"note\": \"n/a\", \"sample_split\": \"validation\", \"sample_id\": \"N147\", \"ai_reference_category\": \"no_useful_information\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S101324\", \"logged_by\": \"SUP-10\", \"note\": \"requested by centre management for load in, signed off\", \"sample_split\": \"validation\", \"sample_id\": \"N148\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104246\", \"logged_by\": \"SUP-13\", \"note\": \"covering for dlamini - booked off sick\", \"sample_split\": \"validation\", \"sample_id\": \"N149\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108451\", \"logged_by\": \"SUP-12\", \"note\": \"took Botha shift as well, 2 posts 1 guard\", \"sample_split\": \"validation\", \"sample_id\": \"N150\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108070\", \"logged_by\": \"SUP-07\", \"note\": \"stood in for Radebe\", \"sample_split\": \"challenge\", \"sample_id\": \"N151\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106978\", \"logged_by\": \"SUP-05\", \"note\": \"next shift akafikanga, ngihlale kuze kube six\", \"sample_split\": \"challenge\", \"sample_id\": \"N152\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103063\", \"logged_by\": \"SUP-22\", \"note\": \"took Wyk shift as well, 2 posts 1 guard\", \"sample_split\": \"challenge\", \"sample_id\": \"N153\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S107032\", \"logged_by\": \"SUP-02\", \"note\": \"oorhandiging was laat, gewag vir sleutels..\", \"sample_split\": \"challenge\", \"sample_id\": \"N154\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S109254\", \"logged_by\": \"SUP-02\", \"note\": \"client says stay till six, dont know if office approved..\", \"sample_split\": \"challenge\", \"sample_id\": \"N155\", \"ai_reference_category\": \"client_requested\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106376\", \"logged_by\": \"SUP-03\", \"note\": \"took Nkosi shift as well, 2 posts 1 guard\", \"sample_split\": \"challenge\", \"sample_id\": \"N156\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S102158\", \"logged_by\": \"SUP-08\", \"note\": \"uTshabalala akezanga namhlanje, ngimele yena\", \"sample_split\": \"challenge\", \"sample_id\": \"N157\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S109296\", \"logged_by\": \"SUP-24\", \"note\": \"NEXT SHIFT AKAFIKANGA, NGIHLALE KUZE KUBE 0600..\", \"sample_split\": \"challenge\", \"sample_id\": \"N158\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106438\", \"logged_by\": \"SUP-06\", \"note\": \"stood in for Mabaso\", \"sample_split\": \"challenge\", \"sample_id\": \"N159\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103831\", \"logged_by\": \"SUP-06\", \"note\": \"uWyk akezanga namhlanje, ngimele yena\", \"sample_split\": \"challenge\", \"sample_id\": \"N160\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S109257\", \"logged_by\": \"SUP-07\", \"note\": \"UDLAMINI AKEZANGA NAMHLANJE, NGIMELE YENA..\", \"sample_split\": \"challenge\", \"sample_id\": \"N161\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S100588\", \"logged_by\": \"SUP-19\", \"note\": \"oorhandiging was laat, gewag vir sletuels\", \"sample_split\": \"challenge\", \"sample_id\": \"N162\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S104379\", \"logged_by\": \"SUP-23\", \"note\": \"uBotha akezanga namhlanje, ngimele yena\", \"sample_split\": \"challenge\", \"sample_id\": \"N163\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108529\", \"logged_by\": \"SUP-11\", \"note\": \"uFourie akezanga namhalnje, ngimele yena\", \"sample_split\": \"challenge\", \"sample_id\": \"N164\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S106753\", \"logged_by\": \"SUP-19\", \"note\": \"took botha shift as well, 2 posts 1 guard\", \"sample_split\": \"challenge\", \"sample_id\": \"N165\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103661\", \"logged_by\": \"SUP-08\", \"note\": \"aflos het nie opgedaag nie, moes aanbly\", \"sample_split\": \"challenge\", \"sample_id\": \"N166\", \"ai_reference_category\": \"relief_problem\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108062\", \"logged_by\": \"SUP-14\", \"note\": \"uBotha akezanga namhlanje, ngimele yena\", \"sample_split\": \"challenge\", \"sample_id\": \"N167\", \"ai_reference_category\": \"absence_cover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103005\", \"logged_by\": \"SUP-08\", \"note\": \"took dlaminni shift as well, 2 posts 1 guard\", \"sample_split\": \"challenge\", \"sample_id\": \"N168\", \"ai_reference_category\": \"cover_unspecified\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S103008\", \"logged_by\": \"SUP-10\", \"note\": \"oorhandiging was laat, gewag vir sleutels\", \"sample_split\": \"challenge\", \"sample_id\": \"N169\", \"ai_reference_category\": \"late_handover\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}, {\"shift_id\": \"S108899\", \"logged_by\": \"SUP-01\", \"note\": \"masjien is stukkend, alles met die hand gedoen\", \"sample_split\": \"challenge\", \"sample_id\": \"N170\", \"ai_reference_category\": \"equipment_failure\", \"reference_provenance\": \"AI-reviewed; not independent human ground truth\"}]")
ai_reference = pd.DataFrame(AI_REFERENCE_ROWS).rename(columns={"note": "reference_note"})
assert ai_reference.shift_id.is_unique
assert set(ai_reference.ai_reference_category) <= set(CATEGORIES)
reference_matches = ai_reference.merge(note_classification_audit, on="shift_id", how="inner",
    validate="one_to_one", suffixes=("_reference", ""))
reference_matches = reference_matches.loc[
    reference_matches.reference_note.eq(reference_matches.note)].copy()
print("Unchanged reference notes present in this export:", len(reference_matches), "/", len(ai_reference))

def reference_agreement(frame, truth_column):
    rows = []
    for split, group in frame.groupby("sample_split"):
        supported = sorted(group[truth_column].unique())
        rows.append({"sample_split": split, "n": len(group),
            "agreement": accuracy_score(group[truth_column], group.category),
            "macro_F1_supported_categories": f1_score(group[truth_column], group.category,
                labels=supported, average="macro", zero_division=0)})
    return pd.DataFrame(rows)

if len(reference_matches):
    print("PROVISIONAL AI-REFERENCE AGREEMENT — not independent human accuracy")
    display(reference_agreement(reference_matches, "ai_reference_category").round(4))
    val = reference_matches.loc[reference_matches.sample_split.eq("validation")].copy()
    if len(val):
        report = classification_report(val.ai_reference_category, val.category,
            labels=sorted(val.ai_reference_category.unique()), output_dict=True, zero_division=0)
        display(pd.DataFrame(report).T.round(3))
        piles = ["client_requested", "operational_associated", "unknown"]
        display(pd.DataFrame(confusion_matrix(val.ai_reference_category.map(CAUSE_PILES),
            val.cause_pile, labels=piles), index=["reference_"+p for p in piles],
            columns=["predicted_"+p for p in piles]))
        known_text = set(reference_matches.loc[reference_matches.sample_split.eq("development"),
            "reference_note"].map(normalise_note))
        print("Validation notes with the same normalised text as a development note:",
              int(val.note.map(normalise_note).isin(known_text).sum()), "/", len(val))
    print("Every disagreement with the provisional reference:")
    with pd.option_context("display.max_colwidth", None):
        display(reference_matches.loc[
            reference_matches.ai_reference_category.ne(reference_matches.category),
            ["sample_split", "shift_id", "note", "ai_reference_category", "category", "typo_corrections"]])

# Blind human review: no classifier or AI-reference category is exported here.
review_template = reference_matches[["sample_id", "sample_split", "shift_id", "reference_note"]].rename(
    columns={"reference_note": "note"}).copy()
review_template["human_category"] = ""
review_template["reviewer"] = ""
review_template["review_comment"] = ""
review_path = STAGE5_OUTPUT / "note_validation_review.csv"
if not review_path.exists():
    review_template.to_csv(review_path, index=False)
else:
    print("Existing human-review CSV preserved; it has not been overwritten.")


Unchanged reference notes present in this export: 170 / 170
PROVISIONAL AI-REFERENCE AGREEMENT — not independent human accuracy
  sample_split    n  agreement  macro_F1_supported_categories
0    challenge   20       1.00                         1.0000
1  development   50       1.00                         1.0000
2   validation  100       0.96                         0.9777
                       precision  recall  f1-score  support
absence_cover                1.0   0.941     0.970     17.0
client_requested             1.0   0.963     0.981     27.0
cover_unspecified            1.0   1.000     1.000      9.0
equipment_failure            1.0   1.000     1.000      7.0
late_handover                1.0   0.875     0.933      8.0
no_useful_information        1.0   1.000     1.000     19.0
relief_problem               1.0   0.923     0.960     13.0
micro avg                    1.0   0.960     0.980    100.0
macro avg                    1.0   0.957     0.978    100.0
weighted avg            

## Independent human validation

Open `note_validation_review.csv`. Before looking at model outputs, fill
`human_category` with an exact taxonomy label, add your name in `reviewer`, and
record uncertainty in `review_comment`. The random validation rows are the main
performance sample; challenge rows assess specific difficult wording separately.
Do not review only obvious successes. Do not infer absence from unspecified cover.

Replace the file in `jem_stage5_outputs` with your reviewed copy and rerun the
next cell. Review inputs are joined by shift ID and checked against exact note
text. Empty human labels are never replaced with AI labels. Partial review is
reported with its coverage and must not be presented as a completed validation.


In [5]:
human_review = pd.read_csv(review_path, dtype="string", keep_default_na=False)
assert human_review.shift_id.is_unique
assert {"shift_id", "note", "human_category", "reviewer"} <= set(human_review.columns)
filled = human_review.human_category.str.strip().ne("")
assert human_review.loc[filled, "human_category"].isin(CATEGORIES).all(), "Unknown human category."
assert human_review.loc[filled, "reviewer"].str.strip().ne("").all(), "Record the reviewer."
# Sample split comes from the frozen reference, not an editable spreadsheet field.
human_scored = human_review.loc[filled, ["shift_id", "note", "human_category", "reviewer"]].merge(
    reference_matches[["shift_id", "sample_split", "note", "category"]], on="shift_id",
    how="inner", validate="one_to_one", suffixes=("_reviewed", "_current"))
assert human_scored.note_reviewed.eq(human_scored.note_current).all(), "A reviewed note has changed."
coverage_human = reference_matches.groupby("sample_split").size().rename("expected").to_frame()
coverage_human["human_reviewed"] = human_scored.groupby("sample_split").size().reindex(
    coverage_human.index, fill_value=0)
display(coverage_human)
if len(human_scored):
    print("Agreement with the supplied HUMAN review; inspect coverage before interpreting.")
    display(reference_agreement(human_scored, "human_category").round(4))
    display(human_scored.loc[human_scored.human_category.ne(human_scored.category)])
else:
    print("No independent human validation completed yet. AI-reference results remain provisional.")


              expected  human_reviewed
sample_split                          
challenge           20               0
development         50               0
validation         100               0
No independent human validation completed yet. AI-reference results remain provisional.


## Associate recorded overtime with note causes and sites

The notes do not specify the exact extra hours they caused, and there is no
planned roster. We therefore cannot calculate a causal or billable-hours split.

Main accounting convention: within each employee-week, assign ordinary hours to
the first 45 recorded hours in chronological order. Allocate subsequent hours to
their shift's note category. Unnoted, routine, ambiguous and unspecified-cover
hours remain unknown. Use the actual shift site, not the employee's primary site.

Include completed, eligible employee-weeks only, preserving the clean-hours
policy. Missing/overlapping employee-weeks are excluded, not treated as zero
overtime. The current partial week is not included in historical overtime totals.

As a sensitivity check, also spread each employee-week's overtime across its
shifts in proportion to recorded shift hours. Neither convention identifies
causality. Different splits show how much the conclusion depends on allocation.
All summaries are provisional until note classifications are independently checked.


In [6]:
eligible_keys = weekly_outcomes.loc[
    weekly_outcomes.label_eligible & weekly_outcomes.week_start.lt(CURRENT_WEEK),
    ["employee_id", "week_start", "outcome_recorded_hours"]]
allocated = shifts.merge(eligible_keys, on=["employee_id", "week_start"], validate="many_to_one")
assert allocated.recorded_hours.notna().all()
allocated = allocated.sort_values(["employee_id", "week_start", "start_at", "shift_id"])
allocated["cumulative_hours"] = allocated.groupby(["employee_id", "week_start"]).recorded_hours.cumsum()
allocated["previous_hours"] = allocated.cumulative_hours - allocated.recorded_hours
allocated["allocated_overtime_hours"] = (
    (allocated.cumulative_hours - 45).clip(lower=0)
    - (allocated.previous_hours - 45).clip(lower=0))
allocated["proportional_overtime_hours"] = (
    (allocated.outcome_recorded_hours - 45).clip(lower=0)
    * allocated.recorded_hours / allocated.outcome_recorded_hours)
allocated = allocated.merge(note_classification_audit[["shift_id", "category", "cause_pile"]],
    on="shift_id", how="left", validate="one_to_one")
allocated["category"] = allocated.category.fillna("no_note")
allocated["cause_pile"] = allocated.cause_pile.fillna("unknown")
assert allocated.allocated_overtime_hours.ge(-1e-9).all()
assert allocated.allocated_overtime_hours.le(allocated.recorded_hours + 1e-9).all()

check = allocated.groupby(["employee_id", "week_start"]).agg(
    assigned=("allocated_overtime_hours", "sum"),
    proportional=("proportional_overtime_hours", "sum"), total=("recorded_hours", "sum"))
assert np.allclose(check.assigned, (check.total - 45).clip(lower=0))
assert np.allclose(check.proportional, check.assigned)

overtime_split = allocated.groupby("cause_pile").agg(
    chronological_hours=("allocated_overtime_hours", "sum"),
    proportional_hours=("proportional_overtime_hours", "sum"))
overtime_split["share_all_overtime"] = overtime_split.chronological_hours / overtime_split.chronological_hours.sum()
print("Overtime ASSOCIATED with categories, not proven causal/billable hours:")
display(overtime_split.round(3))

site_totals = allocated.groupby("site_id").agg(
    recorded_hours=("recorded_hours", "sum"),
    total_overtime_hours=("allocated_overtime_hours", "sum"))
site_piles = allocated.pivot_table(index="site_id", columns="cause_pile",
    values="allocated_overtime_hours", aggfunc="sum", fill_value=0).reindex(
    columns=["client_requested", "operational_associated", "unknown"], fill_value=0)
site_overtime_summary = site_totals.join(site_piles).reset_index().merge(
    sites, on="site_id", validate="one_to_one")
site_overtime_summary["overtime_per_100_recorded_hours"] = (
    100 * site_overtime_summary.total_overtime_hours / site_overtime_summary.recorded_hours)
site_overtime_summary["operational_OT_per_100_recorded_hours"] = (
    100 * site_overtime_summary.operational_associated / site_overtime_summary.recorded_hours)
site_overtime_summary["unknown_share_of_overtime"] = (
    site_overtime_summary.unknown / site_overtime_summary.total_overtime_hours.replace(0, np.nan))
display(site_overtime_summary.sort_values("operational_associated", ascending=False).round(3))
print("Allocated overtime by detailed note category:")
display(allocated.groupby("category").allocated_overtime_hours.sum().sort_values(ascending=False).to_frame())
print("Eligible historical employee-weeks:", len(eligible_keys))
print("Excluded historical employee-weeks:", int((weekly_outcomes.week_start.lt(CURRENT_WEEK)
    & ~weekly_outcomes.label_eligible).sum()))


Overtime ASSOCIATED with categories, not proven causal/billable hours:
                        chronological_hours  proportional_hours  \
cause_pile                                                        
client_requested                     113.75             114.129   
operational_associated               319.50             311.630   
unknown                             1931.25            1938.741   

                        share_all_overtime  
cause_pile                                  
client_requested                     0.048  
operational_associated               0.135  
unknown                              0.817  
  site_id  recorded_hours  total_overtime_hours  client_requested  \
2   ST-03        13322.25                413.00             26.50   
1   ST-02         9943.50                471.00             34.25   
5   ST-06        10011.75                294.75              2.50   
0   ST-01        12291.50                385.50              0.00   
3   ST-04        13096.

## Export and next review

`note_classifications.csv` contains exactly the three assessment-required columns
and every original note. It is the real rule output, including uncertainty labels.
`note_validation_review.csv` is the blind review sheet; existing edits are preserved.
`site_overtime_summary.csv` contains the provisional clean-history allocation.

Retain `note_classification_audit` for original wording, matched evidence, typo
corrections, approval status and review flags. The provisional AI reference is
embedded in this notebook, with its original text and provenance.

Known prototype limitations: four random-sample disagreements were spelling
variants routed to unclear; direct template repetition inflates apparent ease;
the sample has no reference examples of genuinely mixed/unclear causes, so that
class has no demonstrated recall. Approval metadata is not independently scored
by the cause-only review. Unknown hours must not disappear from the denominator.

Complete human review before describing the classifier as independently validated.
If you change rules in response to these validation failures, report the changes
and use a fresh review sample rather than claiming the same sample is untouched.
Do not add note features to the breach model until this stage is reviewed.


In [7]:
classification_path = STAGE5_OUTPUT / "note_classifications.csv"
site_summary_path = STAGE5_OUTPUT / "site_overtime_summary.csv"
note_classifications.to_csv(classification_path, index=False)
site_overtime_summary.to_csv(site_summary_path, index=False)
# Preserve literal placeholders and original text through the CSV round trip.
round_trip = pd.read_csv(classification_path, dtype="string", keep_default_na=False)
assert list(round_trip.columns) == ["shift_id", "category", "note"]
assert round_trip.shift_id.tolist() == notes.shift_id.tolist()
assert round_trip.note.tolist() == notes.note.tolist()
if stage5_frozen_lr is not None:
    pd.testing.assert_frame_equal(lr_predictions, stage5_frozen_lr)
print("Saved:", classification_path, review_path, site_summary_path, sep="\n")
print("Breach predictions were not modified.")


Saved:
jem_stage5_outputs/note_classifications.csv
jem_stage5_outputs/note_validation_review.csv
jem_stage5_outputs/site_overtime_summary.csv
Breach predictions were not modified.
